In [0]:
# Importando a função para gerar o timestamp de ingestão
from pyspark.sql.functions import current_timestamp

# Criando o banco de dados da camada Bronze
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

DataFrame[]

In [0]:
# Caminho do Volume onde os CSVs foram carregados
caminho_volume = "/Volumes/workspace/default/inputs/"

# Mapeamento (Arquivo Original : Nome da Tabela)
tabelas_mapeamento = {
    "movies_info_TMDB_IMDB.csv": "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "bronze.tb_movies_metrics", 
    "credits_and_tags_IMDB_TMDB.csv": "bronze.tb_credits_and_tags",
    "movies_reviews.csv": "bronze.tb_movies_reviews"
}

In [0]:
# Loop para ler cada arquivo, adicionar a coluna e salvar como Delta
for arquivo, nome_tabela in tabelas_mapeamento.items():
    
    # 1. Monta o caminho completo do arquivo
    caminho_arquivo = f"{caminho_volume}{arquivo}"
    
    print(f"Iniciando a carga de: {arquivo} para a tabela {nome_tabela}...")
    
    # 2. Lê os dados do CSV 
    df = spark.read.csv(caminho_arquivo, header=True, inferSchema=True, multiLine=True, escape='"')
    
    # 3. Adiciona a coluna ingestion_datetime contendo o timestamp da inserção
    df_bronze = df.withColumn("ingestion_datetime", current_timestamp())
    
    # 4. Grava a tabela no formato Delta utilizando o modo Append
    df_bronze.write.format("delta").mode("append").saveAsTable(nome_tabela)
    
    print(f"Sucesso! Tabela {nome_tabela} gravada.\n")

Iniciando a carga de: movies_info_TMDB_IMDB.csv para a tabela bronze.tb_movies_info...
Sucesso! Tabela bronze.tb_movies_info gravada.

Iniciando a carga de: movies_financials_IMDB_TMDB.csv para a tabela bronze.tb_movies_financials...
Sucesso! Tabela bronze.tb_movies_financials gravada.

Iniciando a carga de: movies_metrics_IMDB_TMDB.csv para a tabela bronze.tb_movies_metrics...
Sucesso! Tabela bronze.tb_movies_metrics gravada.

Iniciando a carga de: credits_and_tags_IMDB_TMDB.csv para a tabela bronze.tb_credits_and_tags...
Sucesso! Tabela bronze.tb_credits_and_tags gravada.

Iniciando a carga de: movies_reviews.csv para a tabela bronze.tb_movies_reviews...
Sucesso! Tabela bronze.tb_movies_reviews gravada.



In [0]:
import requests
from datetime import datetime, timedelta
from pyspark.sql.functions import current_timestamp

# Calcular dinamicamente as datas sugeridas (hoje e 7 dias atrás)
hoje = datetime.now()
sete_dias_atras = hoje - timedelta(days=7)

# Formatar as datas para o padrão exigido: MM-DD-AAAA
data_fim_default = hoje.strftime("%m-%d-%Y")
data_inicio_default = sete_dias_atras.strftime("%m-%d-%Y")

# Criar os parâmetros (widgets) interativos no topo do notebook
dbutils.widgets.text("data_inicio", data_inicio_default, "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", data_fim_default, "Data Fim (MM-DD-AAAA)")

print("Widgets criados com sucesso! Verifique o topo do notebook.")

Widgets criados com sucesso! Verifique o topo do notebook.


In [0]:
# 1. Capturar os valores inseridos nos widgets
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

# 2. Montar a URL exata, inserindo as variáveis de data
url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

print(f"Buscando cotações de {data_inicio} até {data_fim}...")

# 3. Fazer a requisição HTTP para a API e extrair o JSON
resposta = requests.get(url)
dados_json = resposta.json()

# A API do BC guarda a lista de cotações dentro de uma chave chamada 'value'
lista_cotacoes = dados_json.get("value", [])

# 4. Validar se a API retornou dados antes de tentar gravar
if len(lista_cotacoes) > 0:
    # Converter a lista retornada pela API para um DataFrame do Spark
    df_cotacao = spark.createDataFrame(lista_cotacoes)
    
    # Adicionar a coluna de timestamp exigida para toda a camada Bronze
    df_cotacao_bronze = df_cotacao.withColumn("ingestion_datetime", current_timestamp())
    
    # Gravar a tabela no formato Delta utilizando o modo Append
    nome_tabela_api = "bronze.tb_cotacao_dolar"
    df_cotacao_bronze.write.format("delta").mode("append").saveAsTable(nome_tabela_api)
    
    print(f"Sucesso! Foram gravados {df_cotacao_bronze.count()} registros na tabela {nome_tabela_api}.")
    
    # Mostrar uma prévia da tabela na tela para visualizar como ficou
    display(df_cotacao_bronze)
else:
    print("Nenhuma cotação retornada para este período (pode ser final de semana/feriado).")

Buscando cotações de 09-28-2026 até 10-05-2026...
Sucesso! Foram gravados 6 registros na tabela bronze.tb_cotacao_dolar.


cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.2126,2026-09-28 13:03:11.35885,2026-10-05T21:35:45.462Z
5.2198,2026-09-29 13:06:06.446305,2026-10-05T21:35:45.462Z
5.1803,2026-09-30 13:11:44.783262,2026-10-05T21:35:45.462Z
5.2073,2026-10-01 13:10:35.4469,2026-10-05T21:35:45.462Z
5.2232,2026-10-02 13:03:16.256632,2026-10-05T21:35:45.462Z
4.9853,2026-10-05 13:07:36.558602,2026-10-05T21:35:45.462Z
